# BƯỚC 1: KHÁM PHÁ VÀ PHÂN TÍCH DỮ LIỆU (EXPLORATORY DATA ANALYSIS - EDA)

## Mục tiêu theo đề bài Practice Exercise 3:
1. Đọc và kiểm tra cấu trúc tập dữ liệu khách hàng (`data/raw/marketing_campaign.csv`, định dạng phân tách bằng tab `\t`).
2. Tạo các đặc trưng phân cụm cốt lõi (`Age = 2014 - Year_Birth`, `Total_Spending = sum(6 cột Mnt*)`), ghi chú các giá trị đặc biệt và giữ bản ghi hợp lệ và chia 80% Train / 20% Test trước khi điền trung vị `Income`.
3. Phân tích giá trị khuyết thiếu (Missing values), phát hiện dị biệt (Outliers) bằng phương pháp IQR (Tukey's Fences).
4. Khảo sát hình thái phân phối, quan hệ song biến và biến nhân khẩu học thật (`Education`, `Marital_Status`).

In [ ]:
# Thêm đường dẫn gốc của dự án để import module từ src/
import json
import sys
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    CAMPAIGN_BINARY_COLUMNS,
    MIN_VALID_YEAR_BIRTH,
    MAX_VALID_INCOME,
    NOTEBOOK_FIGURES_DIR,
    NOTEBOOK_METRICS_DIR,
    NOTEBOOK_PROCESSED_DIR,
    NUMERICAL_FEATURES,
    RAW_DATA_PATH,
    REFERENCE_YEAR,
    SPENDING_COMPONENT_COLUMNS,
)
from src.data.loader import load_raw_data, split_data, validate_schema
from src.data.preprocessing import (
    add_derived_features,
    audit_missing_and_outliers,
    clean_data,
    compute_train_medians,
)

NOTEBOOK_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_METRICS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Nạp dữ liệu gốc và kiểm định tính toàn vẹn schema
raw_df = load_raw_data(RAW_DATA_PATH)
schema_info = validate_schema(raw_df)
if not schema_info["is_valid"]:
    raise ValueError(f"Schema không hợp lệ: {schema_info}")
raw_df.head()

In [ ]:
# Kiểm toán giá trị khuyết, ngoại lai (IQR) và dị biệt hạng mục trên dữ liệu gốc trước khi làm sạch
raw_audit_df = audit_missing_and_outliers(raw_df)
raw_audit_df.to_csv(NOTEBOOK_METRICS_DIR / "raw_data_audit.csv", index=False)
print("Giá trị Marital_Status thô:", raw_df["Marital_Status"].value_counts().to_dict())
special_rows = raw_df.loc[
    raw_df["Year_Birth"].lt(MIN_VALID_YEAR_BIRTH) | raw_df["Income"].gt(MAX_VALID_INCOME),
    ["ID", "Year_Birth", "Income"],
]
print("Giá trị đặc biệt được giữ lại để theo dõi:")
special_rows

In [ ]:
# Thống kê đủ 29 cột, tính thống kê số trên các cột định lượng
summary = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing": raw_df.isna().sum(),
    "unique": raw_df.nunique(),
})
numeric = raw_df.select_dtypes(include="number")
for statistic in ("mean", "median", "std", "min", "max", "skew"):
    summary[statistic] = getattr(numeric, statistic)()
summary.to_csv(NOTEBOOK_METRICS_DIR / "raw_column_summary.csv")
summary

In [ ]:
# Khảo sát dữ liệu gốc trước khi điền khuyết và chuẩn hóa
eda_df = add_derived_features(raw_df)
columns = ["Year_Birth", "Age", "Income", *SPENDING_COMPONENT_COLUMNS, "Total_Spending"]
NOTEBOOK_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

for plot_type in ("histogram", "boxplot"):
    fig, axes = plt.subplots(2, 5, figsize=(18, 7), layout="constrained")
    for column, ax in zip(columns, axes.flat):
        values = eda_df[column].dropna()
        if plot_type == "histogram":
            ax.hist(values, bins=30)
        else:
            ax.boxplot(values)
        ax.set_title(column)
    fig.savefig(NOTEBOOK_FIGURES_DIR / f"eda_{plot_type}.png", dpi=150)
    plt.show()
    plt.close(fig)

fig, ax = plt.subplots(layout="constrained")
ax.scatter(eda_df["Income"], eda_df["Total_Spending"], alpha=0.4, s=12)
ax.set(xlabel="Income", ylabel="Total_Spending", title="Thu nhập và tổng chi tiêu")
fig.savefig(NOTEBOOK_FIGURES_DIR / "eda_income_spending.png", dpi=150)
plt.show()
plt.close(fig)

In [ ]:
# Tần suất nhóm nhân khẩu học và phản hồi chiến dịch
category_counts = pd.concat(
    {col: raw_df[col].value_counts(dropna=False) for col in ["Education", "Marital_Status", *CAMPAIGN_BINARY_COLUMNS]},
    names=["Cột", "Giá trị"],
).rename("Số khách hàng")
category_counts.to_csv(NOTEBOOK_METRICS_DIR / "eda_category_counts.csv")
category_counts.to_frame()

In [ ]:
# Chia tập trước khi tính trung vị, giữ thứ tự ID gốc trong bản bàn giao đầy đủ
train_raw, test_raw = split_data(raw_df, save_files=False)
train_medians = compute_train_medians(train_raw)
train_df = clean_data(train_raw, fill_medians=train_medians)
test_df = clean_data(test_raw, fill_medians=train_medians)
df_clean = clean_data(raw_df, fill_medians=train_medians)

assert df_clean["ID"].equals(raw_df["ID"].reset_index(drop=True))
assert set(train_df["ID"]).isdisjoint(test_df["ID"])
train_df.to_csv(NOTEBOOK_PROCESSED_DIR / "train.csv", index=False)
test_df.to_csv(NOTEBOOK_PROCESSED_DIR / "test.csv", index=False)
df_clean.to_csv(NOTEBOOK_PROCESSED_DIR / "full_customers.csv", index=False)

preprocessing_metadata = {
    "reference_year": REFERENCE_YEAR,
    "feature_medians": train_medians,
    "spending_component_columns": SPENDING_COMPONENT_COLUMNS,
}
(NOTEBOOK_PROCESSED_DIR / "preprocessing_metadata.json").write_text(
    json.dumps(preprocessing_metadata, indent=2), encoding="utf-8"
)
audit_missing_and_outliers(train_df).to_csv(
    NOTEBOOK_METRICS_DIR / "train_iqr_audit.csv", index=False
)
